# 📄 Markdown →  PDF Converter
### High-quality PDF generation with consistent styling, syntax highlighting & VS Code-like code blocks
---
> **How to use:**
> 1. Set your **file name** and paste your **Markdown content** in Cell below
> 2. Run all cells
> 3. Click the **Download** button to get your PDF

In [ ]:

FILE_NAME = ""  # PDF file name (without .pdf)

MD_CONTENT = r"""

"""

In [12]:
# ============================================================
#  📦 Install Dependencies
# ============================================================
import subprocess, sys

packages = ["markdown", "Pygments", "weasyprint"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All dependencies installed successfully!")

✅ All dependencies installed successfully!


In [13]:
# ============================================================
#  🎨 PDF Styling Engine + Code Highlighter
# ============================================================

import markdown
from markdown.extensions import Extension
from markdown.treeprocessors import Treeprocessor
from markdown.preprocessors import Preprocessor
from pygments import highlight as pyg_highlight
from pygments.lexers import get_lexer_by_name, guess_lexer, TextLexer
from pygments.formatters import HtmlFormatter
import re
import xml.etree.ElementTree as ET


# ============================================================
#  Extension 0: Auto-insert blank lines before list starts
#  Fixes: "- item" right after "**Bold:**" or "### Heading"
#  without a blank line → markdown treats it as paragraph text
# ============================================================
class ListSpacingPreprocessor(Preprocessor):
    LIST_RE = re.compile(r'^(\s*)([-*+]|\d+[.)]) ')

    def run(self, lines):
        result = []
        for i, line in enumerate(lines):
            if self.LIST_RE.match(line):
                # Check if previous non-blank line is NOT a list item
                prev_line = ''
                for j in range(len(result) - 1, -1, -1):
                    if result[j].strip():
                        prev_line = result[j]
                        break
                if prev_line and not self.LIST_RE.match(prev_line):
                    # Insert blank line before this list start
                    result.append('')
            result.append(line)
        return result


class ListSpacingExtension(Extension):
    def extendMarkdown(self, md):
        md.preprocessors.register(
            ListSpacingPreprocessor(md), 'list_spacing', 120
        )


# ============================================================
#  Extension 1: Yellow highlight on QUESTIONS (Q1, Q46, etc.)
#  Handles TWO formats:
#    a) ## Q1. What is ...   → h2 text starts with Q + digit
#    b) **Q1. What is ...**  → <strong> text starts with Q + digit
#  Section headings like "## System Overview" are NOT touched.
# ============================================================
QUESTION_RE = re.compile(r'^Q\d')

class QuestionHighlighter(Treeprocessor):
    HIGHLIGHT_STYLE = (
        'background-color: #ffff00; '
        'padding: 2px 8px; '
        'border-radius: 4px; '
        'font-weight: 700; '
        'box-decoration-break: clone; '
        '-webkit-box-decoration-break: clone;'
    )

    def _get_full_text(self, element):
        """Get all text content of element (text + children text)."""
        parts = []
        if element.text:
            parts.append(element.text)
        for child in element:
            if child.text:
                parts.append(child.text)
            if child.tail:
                parts.append(child.tail)
        return ''.join(parts).strip()

    def _wrap_in_highlight(self, element):
        """Wrap element's content in a yellow highlight span."""
        span = ET.Element('span')
        span.set('style', self.HIGHLIGHT_STYLE)
        span.text = element.text or ''
        element.text = None
        children = list(element)
        for child in children:
            element.remove(child)
            span.append(child)
        element.append(span)

    def run(self, root):
        # Case A: ## Q1. ... (h2 headings that are questions)
        for element in root.iter('h2'):
            full_text = self._get_full_text(element)
            if QUESTION_RE.match(full_text):
                self._wrap_in_highlight(element)

        # Case B: **Q1. ...** (bold text that is a question)
        # These render as <p><strong>Q1. ...</strong></p>
        for p in root.iter('p'):
            for strong in list(p.iter('strong')):
                if strong == p:
                    continue
                full_text = self._get_full_text(strong)
                if QUESTION_RE.match(full_text):
                    strong.set('style', self.HIGHLIGHT_STYLE)


class QuestionHighlightExtension(Extension):
    def extendMarkdown(self, md):
        md.treeprocessors.register(QuestionHighlighter(md), 'question_highlight', 5)


# ============================================================
#  Extension 2: Inline styles on ALL table elements
# ============================================================
class TableStyler(Treeprocessor):
    TABLE_STYLE = (
        'width: 100%; border-collapse: collapse; margin: 14px 0; '
        'font-size: 10.5pt; page-break-inside: avoid; '
        'border: 2px solid #b8960f;'
    )
    TH_STYLE = (
        'background-color: #e6d270; color: #1a1206; font-weight: 700; '
        'padding: 10px 14px; border: 1px solid #b8960f; text-align: left; '
        'font-size: 10.5pt;'
    )
    TD_EVEN = (
        'padding: 9px 14px; border: 1px solid #d4c090; '
        'background-color: #fff8e1; color: #2c2417; font-size: 10.5pt;'
    )
    TD_ODD = (
        'padding: 9px 14px; border: 1px solid #d4c090; '
        'background-color: #ffedaa; color: #2c2417; font-size: 10.5pt;'
    )

    def run(self, root):
        for table in root.iter('table'):
            table.set('style', self.TABLE_STYLE)
        for th in root.iter('th'):
            th.set('style', self.TH_STYLE)
        for tbody in root.iter('tbody'):
            for i, tr in enumerate(c for c in tbody if c.tag == 'tr'):
                style = self.TD_EVEN if i % 2 == 0 else self.TD_ODD
                for td in (c for c in tr if c.tag == 'td'):
                    td.set('style', style)
        for table in root.iter('table'):
            for i, tr in enumerate(c for c in table if c.tag == 'tr'):
                style = self.TD_EVEN if i % 2 == 0 else self.TD_ODD
                for td in (c for c in tr if c.tag == 'td'):
                    td.set('style', style)


class TableStyleExtension(Extension):
    def extendMarkdown(self, md):
        md.treeprocessors.register(TableStyler(md), 'table_styler', 4)


# ============================================================
#  Extension 3: Fenced code → Pygments INLINE styles
#  FIXED: non-greedy, preserves surrounding blank lines for MD
# ============================================================
class InlineCodeHighlightPreprocessor(Preprocessor):
    # Match fenced blocks: ```lang\ncode\n```
    # Non-greedy, line-anchored, preserves blank lines
    FENCED_RE = re.compile(
        r'^```([\w+-]*)[ \t]*\n(.*?\n)```[ \t]*$',
        re.MULTILINE | re.DOTALL
    )

    PRE_STYLE = (
        'background-color: #1e1e1e; '
        'color: #d4d4d4; '
        'border-radius: 8px; '
        'padding: 16px 20px; '
        'font-family: Consolas, monospace; '
        'font-size: 10.5pt; '
        'line-height: 1.7; '
        'border-left: 4px solid #007acc; '
        'margin: 12px 0 18px 0; '
        'white-space: pre-wrap; '
        'word-wrap: break-word;'
    )

    def run(self, lines):
        text = '\n'.join(lines)

        # Use a non-greedy approach: find each ``` pair one at a time
        result = []
        last_end = 0

        for match in self.FENCED_RE.finditer(text):
            # Add text before this match
            result.append(text[last_end:match.start()])

            lang = match.group(1).strip() or ''
            code = match.group(2)
            if code.endswith('\n'):
                code = code[:-1]

            # Pick lexer
            try:
                lexer = get_lexer_by_name(lang) if lang else TextLexer()
            except Exception:
                try:
                    lexer = guess_lexer(code)
                except Exception:
                    lexer = TextLexer()

            # Highlight with inline styles
            formatter = HtmlFormatter(
                noclasses=True,
                style='monokai',
                wrapcode=True,
                linenos=False,
            )
            highlighted = pyg_highlight(code, lexer, formatter)

            # Inject inline pre styles
            highlighted = highlighted.replace(
                '<pre',
                f'<pre style="{self.PRE_STYLE}"',
                1
            )

            # Wrap with blank lines to preserve markdown block separation
            result.append(f'\n\n{highlighted}\n\n')
            last_end = match.end()

        # Add remaining text
        result.append(text[last_end:])

        return ''.join(result).split('\n')


class InlineCodeHighlightExtension(Extension):
    def extendMarkdown(self, md):
        md.preprocessors.register(
            InlineCodeHighlightPreprocessor(md), 'inline_code_highlight', 110
        )


# ============================================================
#  Extension 4: Inline styles on list items (ul/ol/li)
#  Forces WeasyPrint to render lists properly as block elements
# ============================================================
class ListStyler(Treeprocessor):
    UL_STYLE = (
        'margin: 8px 0 14px 0; padding-left: 28px; '
        'list-style-type: disc; color: #2c2417;'
    )
    OL_STYLE = (
        'margin: 8px 0 14px 0; padding-left: 28px; '
        'list-style-type: decimal; color: #2c2417;'
    )
    LI_STYLE = (
        'display: list-item; margin-bottom: 5px; '
        'line-height: 1.7; color: #2c2417;'
    )

    def run(self, root):
        for ul in root.iter('ul'):
            ul.set('style', self.UL_STYLE)
        for ol in root.iter('ol'):
            ol.set('style', self.OL_STYLE)
        for li in root.iter('li'):
            li.set('style', self.LI_STYLE)


class ListStyleExtension(Extension):
    def extendMarkdown(self, md):
        md.treeprocessors.register(ListStyler(md), 'list_styler', 3)


# ============================================================
#  PDF CSS
# ============================================================
PDF_CSS = """
@page {
    size: A4;
    margin: 25mm 20mm 30mm 20mm;
    background-color: #fff2cc;

    @bottom-center {
        content: "Page " counter(page) " of " counter(pages);
        font-family: Arial, sans-serif;
        font-size: 8pt;
        color: #8b7d6b;
    }
}

body {
    font-family: Arial, 'Segoe UI', Helvetica, sans-serif;
    font-size: 11pt;
    line-height: 1.75;
    color: #2c2417;
    background-color: #fff2cc;
    margin: 0;
    padding: 0;
}

h1 {
    font-family: Georgia, 'Times New Roman', serif;
    font-size: 22pt;
    font-weight: 900;
    text-align: center;
    color: #1a1206;
    background-color: #ffe680;
    border: 3px solid #c4a84a;
    border-radius: 12px;
    padding: 18px 24px;
    margin: 0 0 28px 0;
    letter-spacing: 0.5px;
    page-break-after: avoid;
}

h2 {
    font-family: Arial, sans-serif;
    font-size: 12.5pt;
    font-weight: 700;
    color: #1a1206;
    margin: 28px 0 8px 0;
    padding: 0;
    border: none;
    page-break-after: avoid;
}

h3 {
    font-family: Arial, sans-serif;
    font-size: 11.5pt;
    font-weight: 700;
    color: #3d3019;
    margin: 18px 0 6px 0;
    border-bottom: 1px solid #d4c9a8;
    padding-bottom: 4px;
    page-break-after: avoid;
}

h4, h5, h6 {
    font-family: Arial, sans-serif;
    font-weight: 700;
    color: #4a3c23;
    margin: 14px 0 4px 0;
    page-break-after: avoid;
}

p {
    margin: 6px 0 14px 0;
    text-align: justify;
    color: #2c2417;
}

/* Lists — block-level, proper spacing */
ul {
    display: block;
    margin: 8px 0 14px 0;
    padding-left: 28px;
    list-style-type: disc;
    color: #2c2417;
}
ol {
    display: block;
    margin: 8px 0 14px 0;
    padding-left: 28px;
    list-style-type: decimal;
    color: #2c2417;
}
li {
    display: list-item;
    margin-bottom: 5px;
    line-height: 1.7;
    color: #2c2417;
}
ul ul { list-style-type: circle; margin: 4px 0 4px 0; }
ul ul ul { list-style-type: square; }

blockquote {
    border-left: 4px solid #c4a84a;
    background-color: #f5efc6;
    margin: 14px 0;
    padding: 12px 18px;
    border-radius: 0 8px 8px 0;
    color: #3d3019;
    font-style: italic;
}
blockquote p {
    margin: 4px 0;
}

/* Tables */
table {
    width: 100%;
    border-collapse: collapse;
    margin: 14px 0;
    font-size: 10.5pt;
    page-break-inside: avoid;
    border: 2px solid #b8960f;
}
thead { background-color: #e6d270; }
th {
    background-color: #e6d270;
    color: #1a1206;
    font-weight: 700;
    padding: 10px 14px;
    border: 1px solid #b8960f;
    text-align: left;
}
td {
    padding: 9px 14px;
    border: 1px solid #d4c090;
    background-color: #fff8e1;
    color: #2c2417;
}
tr:nth-child(even) td {
    background-color: #ffedaa;
}

hr {
    border: none;
    height: 2px;
    background-color: #c4a84a;
    margin: 22px 0;
}

a { color: #8b6914; text-decoration: underline; }
strong { color: #1a1206; font-weight: 700; }
em { font-style: italic; }

/* Inline code */
code {
    font-family: Consolas, 'Courier New', monospace;
    font-size: 10pt;
}
p code, li code, td code, th code,
h2 code, h2 span code, h3 code, h4 code,
strong code, em code, a code,
blockquote code, dt code, dd code {
    background-color: #e8e0c8;
    color: #c7254e;
    padding: 2px 7px;
    border-radius: 4px;
    font-size: 10pt;
    border: 1px solid #d4c9a8;
}

/* Code blocks (Pygments output) */
.highlight {
    page-break-inside: avoid;
    margin: 12px 0 18px 0;
}
.highlight pre {
    background-color: #1e1e1e !important;
    color: #d4d4d4;
    border-radius: 8px;
    padding: 16px 20px;
    border-left: 4px solid #007acc;
    font-family: Consolas, 'Courier New', monospace;
    font-size: 10.5pt;
    line-height: 1.7;
    margin: 0;
    white-space: pre-wrap;
    word-wrap: break-word;
}

/* Standalone pre (no highlight wrapper) */
pre {
    background-color: #1e1e1e;
    color: #d4d4d4;
    border-radius: 8px;
    padding: 16px 20px;
    border-left: 4px solid #007acc;
    font-family: Consolas, 'Courier New', monospace;
    font-size: 10.5pt;
    line-height: 1.7;
    margin: 12px 0 18px 0;
    white-space: pre-wrap;
    word-wrap: break-word;
    page-break-inside: avoid;
}

img {
    max-width: 100%;
    height: auto;
    margin: 10px 0;
}

dt { font-weight: 700; color: #1a1206; margin-top: 10px; }
dd { margin-left: 20px; margin-bottom: 8px; color: #2c2417; }
"""

print("✅ Styling engine loaded")

✅ Styling engine loaded


In [14]:
# ============================================================
#  🔨 Generate PDF from Markdown
# ============================================================

import os, platform
from pathlib import Path
from weasyprint import HTML
from IPython.display import display, HTML as IPHTML

# Detect environment and set output dir
if os.name == 'nt':
    OUTPUT_DIR = str(Path.home() / "Downloads")
elif os.path.isdir("/content"):
    OUTPUT_DIR = "/content"
else:
    OUTPUT_DIR = str(Path.home() / "Downloads")

os.makedirs(OUTPUT_DIR, exist_ok=True)

def md_to_styled_html(md_text):
    """Convert markdown to fully styled HTML with all inline styles."""
    html_body = markdown.markdown(
        md_text,
        extensions=[
            ListSpacingExtension(),            # auto blank lines before lists
            InlineCodeHighlightExtension(),    # fenced code → Pygments inline
            QuestionHighlightExtension(),      # h2 yellow highlight text only
            TableStyleExtension(),             # inline table/th/td styles
            ListStyleExtension(),              # inline ul/ol/li styles
            'tables',                          # | table | support
            'toc',                             # [TOC] support
            'sane_lists',                      # proper list handling
            'smarty',                          # smart quotes
            'fenced_code',                     # fallback for any missed fenced code
        ],
    )
    return f"""<!DOCTYPE html>
<html><head><meta charset="UTF-8"><style>{PDF_CSS}</style></head>
<body>{html_body}</body></html>"""

def generate_pdf(file_name, md_content):
    safe_name = re.sub(r'[^\w\s-]', '', file_name).strip().replace(' ', '_') or "document"
    pdf_path = os.path.join(OUTPUT_DIR, f"{safe_name}.pdf")

    styled_html = md_to_styled_html(md_content)

    # Debug: save HTML so you can inspect in browser
    html_path = pdf_path.replace('.pdf', '_debug.html')
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(styled_html)

    HTML(string=styled_html).write_pdf(pdf_path)
    size_kb = os.path.getsize(pdf_path) / 1024
    print(f"✅ PDF generated: {pdf_path} ({size_kb:.1f} KB)")
    print(f"   HTML debug: {html_path}")
    return pdf_path

pdf_file = generate_pdf(FILE_NAME, MD_CONTENT)

DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.002s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.005s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:GPOS dropped
INFO:fontTools.subset:GSUB dropped
DEBUG:f

✅ PDF generated: /content/test_pdf.pdf (76.4 KB)
   HTML debug: /content/test_pdf_debug.html


In [15]:
# ============================================================
#  ⬇️ Download Button
# ============================================================

import base64

with open(pdf_file, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

fname = os.path.basename(pdf_file)

display(IPHTML(f"""
<div style="text-align:center; margin:20px 0;">
    <a href="data:application/pdf;base64,{b64}" download="{fname}"
       style="
        display:inline-block; padding:14px 40px;
        background-color:#c4a84a; color:#fff;
        font-family:Arial,sans-serif; font-size:15px; font-weight:700;
        text-decoration:none; border-radius:8px;
        box-shadow:0 3px 10px rgba(0,0,0,0.15); cursor:pointer;
       ">⬇ Download {fname}</a>
</div>
"""))